In [1]:
%pylab inline
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
from tqdm import trange 
from utils import *


Populating the interactive namespace from numpy and matplotlib


In [2]:
# Vanilla RNN Model 

class RNN(nn.Module):
    def __init__(self, in_size, h_size, out_size):
        super(RNN, self).__init__()
        self.h_size = h_size
        self.rn = nn.RNN(in_size, h_size)
        self.i2o = nn.Linear()
    def forward(self, x, h):
        combined = torch.cat((x, h), 1)
        h = self.i2h(combined)
        x = self.i2o(combined)
        x = F.log_softmax(x, dim=1)
        return x, h
    
    def init_h(self):
        return torch.zeros(1, self.h_size) 
n_hid = 128 
rnn = RNN(n_chars, n_hid, n_categs)


In [3]:
# Training Function 
opt = torch.optim.Adam(rnn.parameters())
loss_function = nn.NLLLoss()
def train(categ_tensor, line_tensor):
    h = rnn.init_h()

    for i in range(line_tensor.size(0)):
        out, h = rnn(line_tensor[i], h)
    
    loss = loss_function(out, categ_tensor)

    # Update Weights 
    opt.zero_grad()
    loss.backward()
    opt.step()

    return out, loss.item()


In [20]:
# Training Loop 
epochs = 10000
steps = epochs / 10 

for i in trange(epochs+1):
    categ, line, categ_tensor, line_tensor = gen_rand_training_example()
    out, loss = train(categ_tensor, line_tensor)

    if i % steps == 0:
        guess, guess_i = categ_from_out(out)
        correct = "CORRECT" if guess == categ else f"WRONG ({categ})"
        print(f"{loss:.3f} {line} / {guess} {correct}")
    



100%|██████████| 10001/10001 [00:09<00:00, 1046.91it/s]0.890 Gagne / French CORRECT



In [2]:
def predict(line):
    with torch.no_grad():
        hidden = rnn.init_h()
        line_tensor = line2tensor(line)
        
        for i in range(line_tensor.size()[0]):
            output, hidden = rnn(line_tensor[i], hidden)
        
        pred, _ = categ_from_out(output)
        return pred
inpt = input("Where is this name from?\n")
predict(inpt)

NameError: name 'rnn' is not defined